<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/04_equilibria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Equilibria
In this exercise, you familiarize yourself the stability of dynamical systems, and ways to identify and classify equilibrium points.



In [ ]:
import abc
from typing import Callable
import jax.numpy as jnp
import matplotlib.pyplot as plt
import functools
import jax


In [ ]:
### Helper functions

# simulate ODE dynamics using Euler's method, x_dot = f(x)
def simulate_ode_dynamics(
    ode_dynamics: Callable[[jnp.ndarray], jnp.ndarray],
    initial_state: float,
    tmax: float,
) -> jnp.ndarray:
    dt = 1e-2
    num_steps = int(tmax / dt)
    ts = jnp.linspace(0, tmax, num_steps)
    xs = [initial_state]
    for i in range(1, num_steps):
        xs.append(xs[-1] + ode_dynamics(xs[-1]) * dt)
    x = jnp.array(xs)
    return x, ts


# simulate ODE dynamics using Euler's method, x_dot = f(x, t)
def simulate_ode_dynamics_time(
    ode_dynamics: Callable[[jnp.ndarray, float], jnp.ndarray],
    initial_state: float,
    tmax: float,
) -> jnp.ndarray:
    """Same as simulate_ode_dynamics, but ode_dynamics takes time as an argument too"""
    dt = 1e-2
    num_steps = int(tmax / dt)
    ts = jnp.linspace(0, tmax, num_steps)
    xs = [initial_state]
    for i in range(1, num_steps):
        xs.append(xs[-1] + ode_dynamics(xs[-1], ts[i - 1]) * dt)
    x = jnp.array(xs)
    return x, ts

## System set up
Let's consider the following 1D ODE dynamics, $\dot{x} = \alpha x + \beta x^3$, $\alpha, \beta \in \mathbb{R}$.

In [ ]:
def cubic_ode(x: jnp.ndarray, alpha: float, beta: float) -> jnp.ndarray:
    return alpha * x + beta * x**3

## 1. Basics
To start simple, let's consider the case where $\alpha=-1, \beta=0$.

$ \dot{x} = -x$

In [ ]:
# Define the specific ODE with alpha = -1.0, beta = 0.0
simple_ode = functools.partial(cubic_ode, alpha=-1.0, beta=0.0)

Let's plot a phase portrait (1D, along a line) for `simple_ode`

In [ ]:
x = jnp.linspace(-5, 5, 11 * 6)
dxdt = simple_ode(x)

# Normalize arrow length for visualization
max_dxdt = jnp.max(jnp.abs(dxdt))
arrow_scale = max_dxdt * 1.2

plt.figure(figsize=(10, 2))
plt.plot(x, jnp.zeros_like(x), "k-", alpha=0.2)  # baseline

for xi, dxi in zip(x[::5], dxdt[::5]):
    plt.arrow(
        xi,
        0,
        0.6 * dxi / arrow_scale,
        0,
        head_width=0.08,
        head_length=0.15,
        fc="b",
        ec="b",
    )

plt.xlabel("x")
plt.ylim(-0.5, 0.5)
plt.yticks([])
plt.title("Flow Field: Velocity Direction and Magnitude")
plt.grid(alpha=0.3)
plt.show()

### (i) Identify all equilibrium points (i.e., all $x$ where $\dot{x} = 0$), and whether they are stable/unstable.

[in-class discussion, student response here]

Now, let's simulate several trajectories from various initial conditions, and see what they look like.

In [ ]:
tmax = 10.0
initial_states = jnp.arange(-5, 5)
traj, ts = jax.vmap(simulate_ode_dynamics, in_axes=[None, 0, None])(
    simple_ode, initial_states, tmax
)

plt.figure(figsize=(6, 4))
[plt.plot(tsi, traji.T, color="tab:blue", alpha=0.5) for tsi, traji in zip(ts, traj)]
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Trajectories")
plt.grid(alpha=0.3, zorder=-5)
plt.tight_layout()


### (ii) What would happen if we instead had $\alpha=1.0, \beta=0.0$?

$ \dot{x} = x $


[student response here]

### (iii) What if $\alpha$ was not a constant, but instead a function of time, such as $\alpha = \sin(t)$?

[student response here]

(a)  Describe the behavior of $\dot{x} = \sin(t)x$. How does the $\sin(t)$ affect, if any, the equilibrium points?

(b) (Hand) Draw the phase line for [-5, 5] at $t=0, \frac{\pi}{4}, \frac{\pi}{2}, \frac{3\pi}{4}$. Include sketch as separate pdf in your submission.

In [ ]:
# Please run this code AFTER, to help verify your answer.
def sine_ode(x, t):
    alpha = jnp.sin(t)
    return alpha * x


tmax = 20.0
initial_states = jnp.arange(-5, 5)
traj, ts = jax.vmap(simulate_ode_dynamics_time, in_axes=[None, 0, None])(
    sine_ode, initial_states, tmax
)

plt.figure(figsize=(6, 4))
[plt.plot(tsi, traji.T, color="tab:blue", alpha=0.5) for tsi, traji in zip(ts, traj)]
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Trajectories")
plt.grid(alpha=0.3, zorder=-5)
plt.tight_layout()


## 2. Nonlinear term

Now consider the case where $\alpha=1, \beta=-1$.

$\dot{x} = x - x^3$

In [ ]:
# Define the specific ODE with alpha = 1.0, beta = -1.0
dynamics_ode = functools.partial(cubic_ode, alpha=1.0, beta=-1.0)

### (i) Hand draw the phase line for the system.

Attach an image as part of your submission.

[in-class discussion, response here]

### (ii) Find the equilibria and classify their stability. Provide a brief justification.

[in-class discussion, student response here]

In [ ]:
# Please run this code AFTER, to help verify your answer.
x = jnp.linspace(-2, 2, 150)
dxdt = dynamics_ode(x)

# Normalize arrow length for visualization
max_dxdt = jnp.max(jnp.abs(dxdt))
arrow_scale = max_dxdt * 1.2

plt.figure(figsize=(10, 2))
plt.plot(x, jnp.zeros_like(x), "k-", alpha=0.2)  # baseline

for xi, dxi in zip(x[::5], dxdt[::5]):
    plt.arrow(
        xi,
        0,
        0.6 * dxi / arrow_scale,
        0,
        head_width=0.03,
        head_length=0.05,
        fc="b",
        ec="b",
    )

plt.xlabel("x")
plt.ylim(-0.4, 0.4)
plt.yticks([])
plt.title("Flow Field: Velocity Direction and Magnitude")
plt.grid(alpha=0.3)
plt.show()

### (iii) Describe what the trajectories would look like when starting at $x_0 = -1.5, -0.5, -0.01, 0.01, 0.5, 1.5$. Provide a brief justification.

[in-class discussion, student response here]

In [ ]:
# Please run this code AFTER, to help verify your answer.

tmax = 10.0
initial_states = jnp.array([-1.5, -0.5, -0.01, 0.01, 0.5, 1.5])
traj, ts = jax.vmap(simulate_ode_dynamics, in_axes=[None, 0, None])(
    dynamics_ode, initial_states, tmax
)
x_eq = jnp.array([-1.0, 0.0, 1.0])

plt.figure(figsize=(6, 4))
[plt.plot(tsi, traji.T, color="tab:blue", alpha=0.5) for tsi, traji in zip(ts, traj)]
[plt.hlines(eq, 0, tmax, color="tab:orange", linestyle="--", alpha=0.7) for eq in x_eq]
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Trajectories")
plt.grid(alpha=0.3, zorder=-5)
plt.tight_layout()


### (iv) How does $\beta < 0$ affect the system's behavior? What happens when $|\beta|$ is large? and when $|\beta|$ is small.

[student response here]